<a href="https://colab.research.google.com/github/marilia-borgo/FATEC-API-6-SEMESTRE/blob/marilia%2Fscore-criticidade/Score_de_Criticidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Guia: Como Atualizar os Dados e Usar o Notebook

Este notebook busca dados oficiais da ANEEL, calcula o Score de Criticidade por conjunto de unidades consumidoras e exibe uma tabela interativa. Para atualizar ou customizar a análise, siga os passos abaixo:

### 1. Atualização da Fonte de Dados
Os dados são baixados automaticamente na **primeira célula de código** a partir do Portal de Dados Abertos da ANEEL. Para trocar a fonte:
1. Acesse o [Portal de Dados Abertos da ANEEL](https://dadosabertos.aneel.gov.br).
2. Procure por **"Indicadores de Continuidade Coletivos"**.
3. Copie o link do CSV de **Indicadores Reais** (Série Histórica) e de **Limites**.
4. Substitua os valores de `url_real` e `url_limite` na primeira célula de código.

### 2. Atualização do Período de Análise
Por padrão, o notebook analisa os anos de **2021 a 2025**. Para alterar, modifique a variável `anos_alvo` na segunda célula de código:
```python
anos_alvo = [2021, 2022, 2023, 2024, 2025]
```

### 3. Como o Score de Criticidade é Calculado
O score é calculado com base no desvio percentual entre o valor **realizado** e o **limite** regulatório da ANEEL para DEC e FEC:
```
Desvio = max(0, ((Realizado - Limite) / Limite) × 100)
Score_Criticidade = Desvio_DEC + Desvio_FEC
```
Conjuntos com score **0** estão dentro do limite. Quanto maior o score, mais crítico é o conjunto.

### 4. Tabela Interativa (PK-15)
A última célula exibe a tabela de criticidade com:
- **Filtro por Distribuidora**: selecione uma distribuidora no dropdown para visualizar apenas seus conjuntos.
- **Paginação**: navegue pelos resultados com os botões *Anterior* e *Próxima* (15 linhas por página).
- **Coloração automática**: verde (score = 0), amarelo (score ≤ 50), vermelho (score > 50).

### 5. Executar Tudo
Para re-executar toda a análise do zero:
1. Vá em **Ambiente de Execução** (Runtime).
2. Clique em **Executar Tudo** (Run all).
> **Atenção:** se os arquivos `aneel_realizado.csv` e `aneel_limite.csv` já existirem no ambiente, o download será pulado automaticamente.

Essas primeiras células São da tarefa PK-14 https://jeanroodrigues.atlassian.net/browse/PK-14

In [ ]:
import pandas as pd
import requests
import os

# URLs diretas e oficiais do portal de Dados Abertos da ANEEL (Série 2020-2029 e Limites)
url_real = "https://dadosabertos.aneel.gov.br/dataset/d5f0712e-62f6-4736-8dff-9991f10758a7/resource/4493985c-baea-429c-9df5-3030422c71d7/download/indicadores-continuidade-coletivos-2020-2029.csv"
url_limite = "https://dadosabertos.aneel.gov.br/dataset/d5f0712e-62f6-4736-8dff-9991f10758a7/resource/fd69e1dd-fd66-4269-b60c-cc0b7eb221b4/download/indicadores-continuidade-coletivos-limite.csv"

arq_real = "aneel_realizado.csv"
arq_limite = "aneel_limite.csv"

# Função para bater na ANEEL e baixar
def baixar_aneel(url, nome):
    if not os.path.exists(nome):
        print(f"Baixando {nome} da ANEEL... (Isso pode levar um minutinho)")
        resposta = requests.get(url)
        with open(nome, 'wb') as f:
            f.write(resposta.content)
        print(f"Download de {nome} belezinha! ✅")
    else:
        print(f"{nome} já tá na mão! Pulando download. ⚡")

baixar_aneel(url_real, arq_real)
baixar_aneel(url_limite, arq_limite)

aneel_realizado.csv já tá na mão! Pulando download. ⚡
aneel_limite.csv já tá na mão! Pulando download. ⚡


In [ ]:
import pandas as pd
import numpy as np

print("Carregando os CSVs no Pandas...")
df_real = pd.read_csv('aneel_realizado.csv', sep=';', encoding='latin1')
df_lim = pd.read_csv('aneel_limite.csv', sep=';', encoding='latin1')

# Filtrando para os últimos 5 anos (2021 a 2025)
anos_alvo = [2021, 2022, 2023, 2024, 2025]
df_real = df_real[df_real['AnoIndice'].isin(anos_alvo)]
df_lim = df_lim[df_lim['AnoLimiteQualidade'].isin(anos_alvo)]

df_real = df_real.rename(columns={'VlrIndiceEnviado': 'Valor_Realizado'})
df_lim = df_lim.rename(columns={'VlrLimite': 'Valor_Limite', 'AnoLimiteQualidade': 'AnoIndice'})

df_real['Valor_Realizado'] = df_real['Valor_Realizado'].astype(str).str.replace(',', '.').astype(float)
df_lim['Valor_Limite'] = df_lim['Valor_Limite'].astype(str).str.replace(',', '.').astype(float)

chaves_join = ['SigAgente', 'DscConjUndConsumidoras', 'SigIndicador', 'AnoIndice']
df_completo = pd.merge(df_real, df_lim, on=chaves_join, how='inner')

# Pivotando para DEC e FEC virarem colunas na mesma linha
# IMPORTANTE: Colocamos o 'AnoIndice' no index para manter o histórico separado por ano!
df_tratado = df_completo.pivot_table(
    index=['SigAgente', 'DscConjUndConsumidoras', 'AnoIndice'],
    columns='SigIndicador',
    values=['Valor_Realizado', 'Valor_Limite'],
    aggfunc={
        'Valor_Realizado': 'sum',  # Soma os 12 meses para dar o total do ano
        'Valor_Limite': 'mean'     # Tira a média para manter o valor original do limite anual
    }
).reset_index()
df_tratado.columns = [f"{col[1]}_{col[0]}" if col[1] else col[0] for col in df_tratado.columns]

df_tratado = df_tratado.rename(columns={
    'SigAgente': 'Distribuidora',
    'DscConjUndConsumidoras': 'Conjunto',
    'AnoIndice': 'Ano',
    'DEC_Valor_Realizado': 'DEC_realizado',
    'DEC_Valor_Limite': 'DEC_limite',
    'FEC_Valor_Realizado': 'FEC_realizado',
    'FEC_Valor_Limite': 'FEC_limite'
})

df_tratado = df_tratado.dropna(subset=['DEC_realizado', 'DEC_limite', 'FEC_realizado', 'FEC_limite'])

print("Dados dos últimos 5 anos tratados e cruzados com sucesso!")
display(df_tratado)

Carregando os CSVs no Pandas...


/tmp/ipykernel_21467/2142477551.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lim = pd.read_csv('aneel_limite.csv', sep=';', encoding='latin1')


Dados dos últimos 5 anos tratados e cruzados com sucesso!


,Distribuidora,Conjunto,Ano,DEC_limite,FEC_limite,DEC_realizado,FEC_realizado
0,AME,ALTO SOLIMÕES,2021.0,75.0,62.0,53.35,72.79
1,AME,ALTO SOLIMÕES,2022.0,75.0,62.0,50.01,66.37
2,AME,ALTO SOLIMÕES,2023.0,75.0,62.0,37.50,41.12
3,AME,ALTO SOLIMÕES,2024.0,75.0,62.0,46.59,42.88
4,AME,ALTO SOLIMÕES,2025.0,70.0,57.0,59.51,39.41


In [ ]:
caminho_saida = 'dados_aneel_prontos_para_calculo.csv'
df_tratado.to_csv(caminho_saida, index=False, sep=';', decimal=',')
print(f"Base redonda salva em: {caminho_saida}")

Base redonda salva em: dados_aneel_prontos_para_calculo.csv


In [ ]:
df_calculo = pd.read_csv(caminho_saida, sep=';', decimal=',')

def calcula_desvio(realizado, limite):
    limite = np.where(limite == 0, np.nan, limite)
    calculo = ((realizado - limite) / limite) * 100
    return np.maximum(0, calculo)

df_calculo['Desvio_DEC'] = calcula_desvio(df_calculo['DEC_realizado'], df_calculo['DEC_limite'])
df_calculo['Desvio_FEC'] = calcula_desvio(df_calculo['FEC_realizado'], df_calculo['FEC_limite'])
df_calculo['Score_Criticidade'] = df_calculo['Desvio_DEC'] + df_calculo['Desvio_FEC']

df_resultado = df_calculo.sort_values(by='Score_Criticidade', ascending=False)

display(df_resultado)

,Distribuidora,Conjunto,Ano,DEC_limite,FEC_limite,DEC_realizado,FEC_realizado,Desvio_DEC,Desvio_FEC,Score_Criticidade
10812,EQUATORIAL GO,CABRIUVA S2,2022.0,12.0,11.0,154.97,31.24,1191.416667,184.000000,1375.416667
11094,EQUATORIAL GO,ITAJA S1,2023.0,16.0,10.0,130.76,47.10,717.250000,371.000000,1088.250000
11125,EQUATORIAL GO,ITUMBIARA VELHA S2,2023.0,14.0,9.0,125.35,25.37,795.357143,181.888889,977.246032
10813,EQUATORIAL GO,CABRIUVA S2,2023.0,12.0,10.0,109.14,26.23,809.500000,162.300000,971.800000
554,CEEE-D,UTE PRESIDENTE MÉDICI,2024.0,15.0,9.0,110.02,39.16,633.466667,335.111111,968.577778
...,...,...,...,...,...,...,...,...,...,...
6946,ELEKTRO,CERQUILHO DOIS,2025.0,7.0,6.0,3.33,2.42,0.000000,0.000000,0.000000
6945,ELEKTRO,CERQUILHO DOIS,2024.0,7.0,6.0,3.39,2.09,0.000000,0.000000,0.000000
6944,ELEKTRO,CERQUILHO DOIS,2023.0,7.0,6.0,3.95,1.92,0.000000,0.000000,0.000000
6975,ELEKTRO,CUNHA,2024.0,17.0,9.0,15.14,5.76,0.000000,0.000000,0.000000


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


A partir daqui é da tarefa PK-15 -> https://jeanroodrigues.atlassian.net/browse/PK-15


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import math

colunas_necessarias = ['Conjunto', 'Distribuidora', 'DEC_realizado', 'FEC_realizado', 'Score_Criticidade']
dados_faltantes = not set(colunas_necessarias).issubset(df_calculo.columns)

if df_calculo.empty or dados_faltantes:
    mensagem_erro = """
    <div style='padding: 15px; background-color: #ffebee; color: #c62828; border-left: 5px solid #c62828; border-radius: 4px;'>
        <strong>Aviso:</strong> Não existe um dado necessário para geração da Tabela de Criticidade para determinados conjuntos no banco de dados.
    </div>
    """
    display(HTML(mensagem_erro))
else:
    df_rankeado = df_calculo.sort_values(by='Score_Criticidade', ascending=False).copy()
    df_rankeado.insert(0, 'Rank', range(1, len(df_rankeado) + 1))

    def aplicar_cores(val):
        if pd.isna(val): return ''
        elif val == 0: return 'background-color: #c8e6c9; color: #1b5e20' # Verde
        elif val <= 50: return 'background-color: #fff9c4; color: #f57f17' # Amarelo
        else: return 'background-color: #ffcdd2; color: #b71c1c' # Vermelho

    estado = {'pagina_atual': 1, 'total_paginas': 1} # Dicionário para guardar o estado da paginação

    lista_distribuidoras = ['Todas'] + sorted(df_rankeado['Distribuidora'].unique().tolist())
    dropdown_dist = widgets.Dropdown(
        options=lista_distribuidoras, value='Todas', description='Distribuidora:',
        style={'description_width': 'initial'}
    )

    btn_prev = widgets.Button(description='< Anterior', disabled=True, button_style='info', tooltip='Página anterior')
    btn_next = widgets.Button(description='Próxima >', disabled=True, button_style='info', tooltip='Próxima página')
    lbl_pagina = widgets.Label(value='Página 1 de 1')

    output_tabela = widgets.Output()

    def renderizar_tabela(change=None):
        if change and change.get('owner') == dropdown_dist:
            estado['pagina_atual'] = 1

        output_tabela.clear_output(wait=True)

        with output_tabela:
            dist_selecionada = dropdown_dist.value

            if dist_selecionada == 'Todas':
                df_filtrado = df_rankeado.copy()
            else:
                df_filtrado = df_rankeado[df_rankeado['Distribuidora'] == dist_selecionada].copy()

            total_linhas = len(df_filtrado)

            if total_linhas == 0:
                display(HTML("<p>Nenhum conjunto encontrado para esta distribuidora.</p>"))
                btn_prev.disabled = True
                btn_next.disabled = True
                lbl_pagina.value = 'Página 0 de 0'
                return

            tamanho_pagina = 15
            estado['total_paginas'] = math.ceil(total_linhas / tamanho_pagina)

            if estado['pagina_atual'] > estado['total_paginas']:
                estado['pagina_atual'] = estado['total_paginas']
            btn_prev.disabled = (estado['pagina_atual'] <= 1)
            btn_next.disabled = (estado['pagina_atual'] >= estado['total_paginas'])

            lbl_pagina.value = f"  Página {estado['pagina_atual']} de {estado['total_paginas']}  "

            inicio = (estado['pagina_atual'] - 1) * tamanho_pagina
            fim = inicio + tamanho_pagina
            df_pagina = df_filtrado.iloc[inicio:fim]

            tabela_estilizada = (
                df_pagina.style
                .map(aplicar_cores, subset=['Score_Criticidade'])
                .format({'Score_Criticidade': '{:.2f}', 'Desvio_DEC': '{:.2f}', 'Desvio_FEC': '{:.2f}'})
                .set_properties(**{'text-align': 'center', 'border': '1px solid #ddd'})
                .set_table_styles([{'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('text-align', 'center')]}])
                .hide(axis="index")
            )

            display(tabela_estilizada)
            print(f"Total de conjuntos encontrados: {total_linhas}")

    def ir_para_anterior(b):
        if estado['pagina_atual'] > 1:
            estado['pagina_atual'] -= 1
            renderizar_tabela()

    def ir_para_proxima(b):
        if estado['pagina_atual'] < estado['total_paginas']:
            estado['pagina_atual'] += 1
            renderizar_tabela()

    btn_prev.on_click(ir_para_anterior)
    btn_next.on_click(ir_para_proxima)
    dropdown_dist.observe(renderizar_tabela, names='value')

    controles = widgets.HBox(
        [dropdown_dist, btn_prev, lbl_pagina, btn_next],
        layout=widgets.Layout(align_items='center', margin='0 0 10px 0')
    )

    display(controles)
    display(output_tabela)

    renderizar_tabela()

Output()